# Core machine learning · First steps

**Level** 1 · **Time** about 45 minutes

We will explore some core concepts of machine learning, starting with the fundamental question: What even is machine learning and why do we do it?
By the end of this lesson, you will hopefully see the vision of machine learning as a tool to help us make better decisions and predictions based on data.

>To learn from this notebook, you just have to run it. You do not need to change or understand any of the code. But if you do want to understand it better, you are welcome to explore and experiment with it.

## Help a busy student

Marc is an archeology student. Working with his professor, they discovered hundreds of pottery fragments in a dig site. Preliminary analysis showed that the fragments belong to two ancient Roman vases, which we will call Vase A and Vase B. The professor wants to know which fragments belong to which vase, so she can reconstruct the vases, but she is too busy to do the work herself. Each fragment has to be inspected under a microscope to determine which vase it belongs to. So the professor passes this tedious and time-consuming task to Marc. Today is Friday, and the professor wants the fragments sorted by Monday, so she can start reconstructing the vases. 

But Marc is busy. He already has plans for the weekend, as he is going to a music festival with his friends. In order to quickly classify the fragments, he sorts only 100 of them and then plans on using a machine learning model to classify the rest 400. He assumes that the position of the fragments in the dig site must be a good indicator of their type, as fragments from the same vase are likely to be found in similar locations, so he also records the coordinates of each fragment. 

The problem is that Marc is an archeology student, so he knows nothing about machine learning, so he asks you to help him. Help him as best as you can, knowing that the more you will be wrong, the more will Marc be in trouble with his professor.

In [ ]:
# Section for the imports. Imports are libraries that contain useful functions and classes that we can use in our code. 
import numpy as np                  # Numpy library for numerical computing
import matplotlib.pyplot as plt     # Matplotlib library for plotting
import ipywidgets as widgets
from IPython.display import display
from sklearn.datasets import make_moons                 # Scikit-learn library is the major machine learning library in Python. You will use it a lot.
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

SEED = 42
CLASS_COLORS = ["#386cb0", "#f0027f"]
X, y = make_moons(n_samples=100, noise=0.3, random_state=SEED)  # This function generates the data for you. X is a 2D array of shape (100, 2) 
                                                                # containing the coordinates of the fragments, and y is a 1D array of shape (100,) 
                                                                # containing the labels (0 or 1) for each fragment.
classes = ["Vase A", "Vase B"]      # Fragment classes. Class 0 is Vase A and class 1 is Vase B.

# First we define a function to plot the dataset. 
def plot_dataset(X, y, ax=None, marker="o", alpha=0.8):
    """Function to plot the dataset."""
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 5))

    for label in (0, 1):
        ax.scatter(X[y == label, 0], X[y == label, 1], c=CLASS_COLORS[label],
                    label=classes[label], alpha=alpha, marker=marker)
        
    ax.set(xlabel="x", ylabel="y")
    ax.legend(loc="best", fontsize=8)
    return ax

plot_dataset(X, y)      # Now we call this function to plot it.
plt.title("Position of pottery fragments in the dig site and the vase they belong to")      # We add a title to the plot.
plt.grid()
plt.show()  # And finally we show the plot.

In [ ]:
# These are some additional helper functions that will be useful thoughout the notebook.
def plot_decision_boundary(model, X, y, ax=None, title=None, marker="o", alpha=0.8):
    """Function to plot the data and the decision boundary of a model."""
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 5))

    # First we constuct a grid of points
    x0 = np.linspace(X[:, 0].min() - .35, X[:, 0].max() + .35, 250)
    x1 = np.linspace(X[:, 1].min() - .35, X[:, 1].max() + .35, 250)
    xx, yy = np.meshgrid(x0, x1)

    # Then we evaluate the model on every point in the grid to see which class model predicts it.
    prediction = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    # We plot all the points in the grid, coloring them according to the predicted class.
    ax.contourf(xx, yy, prediction, levels=[-.5, .5, 1.5], colors=[CLASS_COLORS[0], CLASS_COLORS[1]], alpha=.18)
    if title:
        ax.set_title(title)

    # Finally we plot the original data points on top to see how well the model did.
    plot_dataset(X, y, ax=ax, marker=marker, alpha=alpha)

    return ax

def evaluate_model(model):
    """Fit the model on data and return it with its accuracy score."""
    model.fit(X, y)                     # This fits the model to the data. 
    predictions = model.predict(X)      # This makes predictions on the data using the fitted model.
    return {
        "model": model,
        "accuracy": accuracy_score(y, predictions),
    }

def show_scores(scores):
    print(f"Accuracy: {scores['accuracy']:.1%}")

Before using a model, try to make a rule yourself. Click and drag across the plot to draw a boundary with your mouse. The accuracy of your classification using your boundary will be displayed on top of the plot. With **reset**, you can clear the plot and try again. 

In [ ]:
%matplotlib widget

plt.ioff()

# Close previous figure when rerunning the cell
try:
    plt.close(_draw_fig)
except NameError:
    pass

# --------------------------------------------------
# 1. Figure and widgets
# --------------------------------------------------

_draw_fig, ax = plt.subplots(figsize=(8, 6))
fig = _draw_fig

plot_dataset(X, y, ax=ax)
ax.set_title("Draw a decision boundary")

button = widgets.Button(description="Reset", button_style="warning")
score = widgets.HTML(value="Draw a curve and release to evaluate it.")


# --------------------------------------------------
# 2. Drawing state and plot elements
# --------------------------------------------------

drawn_points = []
drawing = False

boundary_line, = ax.plot(
    [], [],
    color="black",
    linewidth=2.5,
    label="Your boundary",
    zorder=5
)

mistake_points = ax.scatter(
    [], [],
    facecolors="none",
    edgecolors="black",
    s=110,
    linewidths=1.5,
    label="Mistakes",
    zorder=6
)


# --------------------------------------------------
# 3. Classification background grid
# --------------------------------------------------

xlim = ax.get_xlim()
ylim = ax.get_ylim()

xx, yy = np.meshgrid(
    np.linspace(*xlim, 120),
    np.linspace(*ylim, 120)
)

grid_points = np.column_stack([xx.ravel(), yy.ravel()])
class_colors = ["#4C72B0", "#DD8452"]

background = None


# --------------------------------------------------
# 4. Classify points using the drawn polyline
# --------------------------------------------------

def classify_points(points, data):
    points = np.asarray(points)
    data = np.asarray(data)

    A = points[:-1]
    segments = points[1:] - points[:-1]

    valid = np.sum(segments**2, axis=1) > 1e-12
    A = A[valid]
    segments = segments[valid]

    if len(A) == 0:
        return None, None

    diff = data[:, None, :] - A[None, :, :]
    lengths_sq = np.sum(segments**2, axis=1)

    t = np.sum(diff * segments[None, :, :], axis=2) / lengths_sq[None, :]
    t = np.clip(t, 0, 1)

    closest = A[None, :, :] + t[:, :, None] * segments[None, :, :]
    distances_sq = np.sum((data[:, None, :] - closest)**2, axis=2)

    nearest = np.argmin(distances_sq, axis=1)
    nearest_segments = segments[nearest]

    nearest_vectors = data - closest[np.arange(len(data)), nearest]

    signed_distance = (
        nearest_segments[:, 0] * nearest_vectors[:, 1]
        - nearest_segments[:, 1] * nearest_vectors[:, 0]
    )

    predictions_1 = (signed_distance > 0).astype(int)
    predictions_2 = (signed_distance < 0).astype(int)

    return predictions_1, predictions_2


# --------------------------------------------------
# 5. Update classification background
# --------------------------------------------------

def update_background(predictions):
    global background

    if background is not None:
        background.remove()

    Z = predictions.reshape(xx.shape)

    background = ax.contourf(
        xx, yy, Z,
        levels=[-0.5, 0.5, 1.5],
        colors=class_colors,
        alpha=0.20,
        zorder=0
    )


# --------------------------------------------------
# 6. Evaluate boundary
# --------------------------------------------------

def evaluate_boundary():
    if len(drawn_points) < 2:
        score.value = "Draw a longer curve."
        return

    points = np.asarray(drawn_points)

    predictions_1, predictions_2 = classify_points(points, X)

    if predictions_1 is None:
        return

    accuracy_1 = accuracy_score(y, predictions_1)
    accuracy_2 = accuracy_score(y, predictions_2)

    if accuracy_1 >= accuracy_2:
        predictions = predictions_1
        accuracy = accuracy_1
        grid_predictions, _ = classify_points(points, grid_points)
    else:
        predictions = predictions_2
        accuracy = accuracy_2
        _, grid_predictions = classify_points(points, grid_points)

    mistakes = predictions != y
    mistake_points.set_offsets(np.asarray(X)[mistakes])

    update_background(grid_predictions)

    score.value = (
        f"<b>Accuracy: {accuracy:.1%}</b> "
        f"· {np.count_nonzero(mistakes)} mistakes"
    )

    fig.canvas.draw_idle()


# --------------------------------------------------
# 7. Mouse interaction
# --------------------------------------------------

def on_press(event):
    global drawing, background

    if event.inaxes != ax or event.xdata is None or event.ydata is None:
        return

    drawing = True
    drawn_points.clear()
    drawn_points.append((event.xdata, event.ydata))

    mistake_points.set_offsets(np.empty((0, 2)))

    if background is not None:
        background.remove()
        background = None

    score.value = "Release the mouse to evaluate your boundary."


def on_motion(event):
    if not drawing or event.inaxes != ax:
        return

    if event.xdata is None or event.ydata is None:
        return

    drawn_points.append((event.xdata, event.ydata))

    points = np.asarray(drawn_points)
    boundary_line.set_data(points[:, 0], points[:, 1])

    fig.canvas.draw_idle()


def on_release(event):
    global drawing

    if not drawing:
        return

    drawing = False
    evaluate_boundary()


# --------------------------------------------------
# 8. Reset
# --------------------------------------------------

def reset_boundary(_):
    global background, drawing

    drawing = False
    drawn_points.clear()

    boundary_line.set_data([], [])
    mistake_points.set_offsets(np.empty((0, 2)))

    if background is not None:
        background.remove()
        background = None

    score.value = "Draw a curve and release to evaluate it."

    fig.canvas.draw_idle()


# --------------------------------------------------
# 9. Connect events and display
# --------------------------------------------------

fig.canvas.mpl_connect("button_press_event", on_press)
fig.canvas.mpl_connect("motion_notify_event", on_motion)
fig.canvas.mpl_connect("button_release_event", on_release)

button.on_click(reset_boundary)

ax.grid()

display(widgets.VBox([
    widgets.HBox([button, score]),
    fig.canvas
]))

Drawing a boundary is a good way to separate the two classes, but it is manual labor. There is no "machine learning" about it. So instead of drawing a boundary ourself, we would like a computer to learn the best boundary possible. 

First we choose a model and then we instruct the computer to learn the best boundary for that model. Don't worry if you don't know what a model is. For now, you can think of it as a set of rules that the computer can use to separate the two classes. 

The first model we can try is using a straight line to separate the two classes. The two rules that we can control are therefore the slope and the intercept of the line. Run the code below and try to find the best straight line that separates the two classes. You can change the slope and intercept of the line using the sliders.

In [ ]:
%matplotlib widget

plt.ioff()

# Close previous figure when rerunning the cell
try:
    plt.close(_line_fig)
except NameError:
    pass

# --------------------------------------------------
# 1. Figure
# --------------------------------------------------

_line_fig, ax = plt.subplots(figsize=(8, 6))
fig = _line_fig

plot_dataset(X, y, ax=ax)
ax.set_title("Linear decision boundary")

xlim = ax.get_xlim()
ylim = ax.get_ylim()

xx, yy = np.meshgrid(
    np.linspace(*xlim, 150),
    np.linspace(*ylim, 150)
)

grid_points = np.column_stack([xx.ravel(), yy.ravel()])
class_colors = ["#4C72B0", "#DD8452"]

background = None

boundary_line, = ax.plot(
    [], [],
    color="black",
    linewidth=2.5,
    label="Decision boundary",
    zorder=5
)

mistake_points = ax.scatter(
    [], [],
    facecolors="none",
    edgecolors="black",
    s=110,
    linewidths=1.5,
    label="Misclassified points",
    zorder=6
)


# --------------------------------------------------
# 2. Sliders and accuracy display
# --------------------------------------------------

slope_slider = widgets.FloatSlider(
    value=0.0,
    min=-5.0,
    max=5.0,
    step=0.1,
    description="Slope (m):",
    continuous_update=True,
    readout_format=".1f"
)

intercept_slider = widgets.FloatSlider(
    value=0.0,
    min=ylim[0],
    max=ylim[1],
    step=0.1,
    description="Intercept (b):",
    continuous_update=True,
    readout_format=".1f"
)

score = widgets.HTML(
    value="Adjust the sliders to find a good boundary."
)


# --------------------------------------------------
# 3. Classification
# --------------------------------------------------

def classify(data, slope, intercept):
    distances = data[:, 1] - (slope * data[:, 0] + intercept)

    predictions_1 = (distances > 0).astype(int)
    predictions_2 = (distances < 0).astype(int)

    return predictions_1, predictions_2


# --------------------------------------------------
# 4. Update visualization
# --------------------------------------------------

def update(change=None):
    global background

    m = slope_slider.value
    b = intercept_slider.value

    # Update boundary line
    x_line = np.array(xlim)
    y_line = m * x_line + b

    boundary_line.set_data(x_line, y_line)

    # Evaluate both possible class orientations
    pred_1, pred_2 = classify(X, m, b)

    acc_1 = accuracy_score(y, pred_1)
    acc_2 = accuracy_score(y, pred_2)

    if acc_1 >= acc_2:
        predictions = pred_1
        accuracy = acc_1
        orientation = 1
    else:
        predictions = pred_2
        accuracy = acc_2
        orientation = 2

    # Highlight mistakes
    mistakes = predictions != y
    mistake_points.set_offsets(np.asarray(X)[mistakes])

    # Update classification regions
    if background is not None:
        background.remove()

    grid_pred_1, grid_pred_2 = classify(grid_points, m, b)

    grid_predictions = grid_pred_1 if orientation == 1 else grid_pred_2

    background = ax.contourf(
        xx,
        yy,
        grid_predictions.reshape(xx.shape),
        levels=[-0.5, 0.5, 1.5],
        colors=class_colors,
        alpha=0.20,
        zorder=0
    )

    # Update score
    score.value = (
        f"<b>Accuracy: {accuracy:.1%}</b>"
        f" · {np.count_nonzero(mistakes)} mistakes"
        f"<br><small>Decision boundary: "
        f"y = {m:.1f}x + {b:.1f}</small>"
    )

    fig.canvas.draw_idle()


# --------------------------------------------------
# 5. Connect sliders
# --------------------------------------------------

slope_slider.observe(update, names="value")
intercept_slider.observe(update, names="value")

ax.grid()

# Initial visualization
update()


# --------------------------------------------------
# 6. Display everything once
# --------------------------------------------------

display(widgets.VBox([
    widgets.VBox([
        slope_slider,
        intercept_slider,
        score
    ]),
    fig.canvas
]))

Are you satisfied with the results? Hopefully you achieved a good accuracy. The model you have just fitted is an example of **logistic regression**. As you can see, a straight line is not enough to separate the two classes. Luckily, logistic regression can be extended to use a quadratic, qubic, or even higher order polynomial to separate the two classes. Run the code below and change the degree of the polynomial to see how it affects the accuracy.

In [ ]:
%matplotlib widget

plt.ioff()

try:
    plt.close(_logistic_fig)
except NameError:
    pass


# --------------------------------------------------
# 1. Model
# --------------------------------------------------

def create_polynomial_logistic_regression(degree):
    """Create a polynomial logistic regression model."""
    return make_pipeline(
        PolynomialFeatures(degree=degree),
        StandardScaler(),
        LogisticRegression(max_iter=2000, random_state=SEED)
    )


# --------------------------------------------------
# 2. Figure and widgets
# --------------------------------------------------

_logistic_fig, ax = plt.subplots(figsize=(7, 5))

degree_slider = widgets.IntSlider(
    value=1,
    min=1,
    max=12,
    step=1,
    description="Degree:",
    continuous_update=True
)

score = widgets.HTML()


# --------------------------------------------------
# 3. Update visualization
# --------------------------------------------------

def show_logistic(degree=1):

    ax.clear()

    model = create_polynomial_logistic_regression(degree)
    scores = evaluate_model(model)

    plot_decision_boundary(
        model,
        X,
        y,
        ax=ax,
        title=f"Polynomial degree {degree}"
    )

    score.value = f"<b>Accuracy: {scores['accuracy']:.1%}</b>"

    _logistic_fig.canvas.draw_idle()


# --------------------------------------------------
# 4. Connect slider
# --------------------------------------------------

degree_slider.observe(
    lambda change: show_logistic(change["new"]),
    names="value"
)


# --------------------------------------------------
# 5. Initial visualization and display
# --------------------------------------------------

show_logistic(degree_slider.value)

display(widgets.VBox([
    degree_slider,
    score,
    _logistic_fig.canvas
]))

Congratulations! You have just trained your first machine learning model. By changing the polynomial degree, you changed the complexity of the model. In the background, the computer automatically learned the optimal parameters.

But logistic regression is not the only model out there. In the next section, you can choose between **K-nearest neighbors**, **decision trees**, **support vector machines**, **neural networks**, and of course **logistic regression**. We will learn about all of these models throughout the course, but for now, you can just try them out and see how they perform. Write down the settings that gave you the best accuracy for each model. It will come in handy later on.

Based on your experiments, which model and settings would you recommend to Marc to classify the fragments?

In [ ]:
%matplotlib widget
plt.ioff()

try:
    plt.close(_model_fig)
except NameError:
    pass


model_names = [
    "Logistic regression",
    "K-nearest neighbors",
    "Decision tree",
    "SVM (RBF)",
    "Neural network",
]

model_choice = widgets.Dropdown(
    options=model_names,
    description="model"
)

degree = widgets.IntSlider(value=3, min=1, max=12, description="degree")
neighbors = widgets.IntSlider(value=5, min=1, max=30, description="neighbors")
tree_depth = widgets.IntSlider(value=5, min=1, max=15, description="max depth")

svm_c = widgets.FloatLogSlider(
    value=1, base=10, min=-2, max=3, step=.1, description="C"
)
svm_gamma = widgets.FloatLogSlider(
    value=.1, base=10, min=-3, max=1, step=.1, description="gamma"
)

layers = widgets.IntSlider(value=2, min=1, max=3, description="hidden layers")
neurons = widgets.IntSlider(value=20, min=5, max=60, step=5, description="neurons/layer")

controls = [
    degree,
    neighbors,
    tree_depth,
    svm_c,
    svm_gamma,
    layers,
    neurons,
]

control_box = widgets.VBox(controls)
score = widgets.HTML()


def selected_model():
    if model_choice.value == "Logistic regression":
        return create_polynomial_logistic_regression(degree.value), [degree]

    if model_choice.value == "K-nearest neighbors":
        model = make_pipeline(
            StandardScaler(),
            KNeighborsClassifier(n_neighbors=neighbors.value)
        )
        return model, [neighbors]

    if model_choice.value == "Decision tree":
        return DecisionTreeClassifier(
            max_depth=tree_depth.value,
            random_state=SEED
        ), [tree_depth]

    if model_choice.value == "SVM (RBF)":
        model = make_pipeline(
            StandardScaler(),
            SVC(
                C=svm_c.value,
                gamma=svm_gamma.value
            )
        )
        return model, [svm_c, svm_gamma]

    hidden = tuple([neurons.value] * layers.value)
    model = make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=hidden,
            max_iter=2000,
            random_state=SEED
        )
    )
    return model, [layers, neurons]


_model_fig, ax = plt.subplots(figsize=(8, 5))


def update_playground(change=None):
    model, active = selected_model()

    for control in controls:
        control.layout.display = "" if control in active else "none"

    scores = evaluate_model(model)

    ax.clear()
    plot_decision_boundary(
        model, X, y,
        ax=ax,
        title=model_choice.value
    )

    score.value = f"<b>Accuracy: {scores['accuracy']:.1%}</b>"
    _model_fig.canvas.draw_idle()


model_choice.observe(update_playground, names="value")

for control in controls:
    control.observe(update_playground, names="value")

update_playground()

display(widgets.VBox([
    model_choice,
    control_box,
    score,
    _model_fig.canvas
]))

>For the curious: Decision trees model makes completly vertical or horizontal boundaries. Can you figure out why? 

### After the weekend

It's Monday morning. Based on your recommendations, Marc used the model and settings you suggested to classify the rest of the 400 fragments. Without knowing the true labels of the fragments, he reconstructed the two vases. But when he showed the reconstructed vases to his professor, she was not happy. She said that the vases were not reconstructed correctly and that Marc must have misclassified some of the fragments.

He and his professor then had to manually reclassify all the fragments. Marc later gave you the remaining 400 fragments and their true labels. Play again with the same models and settings you used before and see how well they perform on the new data. Does higher accuracy on the original 100 fragments mean higher accuracy on the new 400 fragments?

In [ ]:
%matplotlib widget
plt.ioff()

try:
    plt.close(_model_fig)
except NameError:
    pass


model_names = [
    "Logistic regression",
    "K-nearest neighbors",
    "Decision tree",
    "SVM (RBF)",
    "Neural network",
]

model_choice = widgets.Dropdown(
    options=model_names,
    description="model"
)

degree = widgets.IntSlider(value=3, min=1, max=12, description="degree")
neighbors = widgets.IntSlider(value=5, min=1, max=30, description="neighbors")
tree_depth = widgets.IntSlider(value=5, min=1, max=15, description="max depth")

svm_c = widgets.FloatLogSlider(
    value=1, base=10, min=-2, max=3, step=.1, description="C"
)
svm_gamma = widgets.FloatLogSlider(
    value=.1, base=10, min=-3, max=1, step=.1, description="gamma"
)

layers = widgets.IntSlider(value=2, min=1, max=3, description="hidden layers")
neurons = widgets.IntSlider(value=20, min=5, max=60, step=5, description="neurons/layer")

controls = [
    degree,
    neighbors,
    tree_depth,
    svm_c,
    svm_gamma,
    layers,
    neurons,
]

control_box = widgets.VBox(controls)
score = widgets.HTML()


def selected_model():
    if model_choice.value == "Logistic regression":
        return create_polynomial_logistic_regression(degree.value), [degree]

    if model_choice.value == "K-nearest neighbors":
        model = make_pipeline(
            StandardScaler(),
            KNeighborsClassifier(n_neighbors=neighbors.value)
        )
        return model, [neighbors]

    if model_choice.value == "Decision tree":
        return DecisionTreeClassifier(
            max_depth=tree_depth.value,
            random_state=SEED
        ), [tree_depth]

    if model_choice.value == "SVM (RBF)":
        model = make_pipeline(
            StandardScaler(),
            SVC(
                C=svm_c.value,
                gamma=svm_gamma.value
            )
        )
        return model, [svm_c, svm_gamma]

    hidden = tuple([neurons.value] * layers.value)
    model = make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=hidden,
            max_iter=2000,
            random_state=SEED
        )
    )
    return model, [layers, neurons]


_model_fig, ax = plt.subplots(figsize=(8, 5))

X_new, y_new = make_moons(n_samples=200, noise=0.3, random_state=SEED + 1)

def update_playground(change=None):
    model, active = selected_model()

    for control in controls:
        control.layout.display = "" if control in active else "none"

    scores = evaluate_model(model)

    ax.clear()
    plot_decision_boundary(model, X, y, ax=ax, title=model_choice.value, marker="o", alpha=0.5)

    predictions = model.predict(X_new)
    plot_dataset(X_new, y_new, ax=ax, marker="x", alpha=0.3)
    test_accuracy = accuracy_score(y_new, predictions)

    score.value = f"<b>Accuracy on original data: {scores['accuracy']:.1%}</b><br><b> Accuracy on new data: {test_accuracy:.1%}</b>"
    _model_fig.canvas.draw_idle()


model_choice.observe(update_playground, names="value")

for control in controls:
    control.observe(update_playground, names="value")

update_playground()

display(widgets.VBox([
    model_choice,
    control_box,
    score,
    _model_fig.canvas
]))